In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
#Load daily returns
PROCESSED_DIR = Path("../data/processed")
returns = pd.read_csv(PROCESSED_DIR / "returns_daily.csv", index_col="Date", parse_dates=True)

In [3]:
# If you saved benchmark returns separately, load it here.
# Otherwise, generate benchmark returns from raw ^GSPC in Phase 3 style.
RAW_DIR = Path("../data/raw")
gspc = pd.read_csv(RAW_DIR / "^GSPC_raw.csv")
gspc["Date"] = pd.to_datetime(gspc["Date"], errors="coerce")
gspc = gspc[["Date", "Adj Close"]].rename(columns={"Adj Close":"^GSPC"}).sort_values("Date")
gspc["^GSPC"] = pd.to_numeric(gspc["^GSPC"], errors="coerce")
gspc["^GSPC_ret"] = gspc["^GSPC"].pct_change(fill_method=None)
gspc = gspc.dropna()
gspc.head()

,Date,^GSPC,^GSPC_ret
2,2014-01-03,1831.369995,-0.000333
3,2014-01-06,1826.770020,-0.002512
4,2014-01-07,1837.880005,0.006082
5,2014-01-08,1837.489990,-0.000212
6,2014-01-09,1838.130005,0.000348


In [4]:
# Align benchmark to returns dates
bench = gspc.set_index("Date").reindex(returns.index)["^GSPC_ret"].dropna()
common_idx = returns.index.intersection(bench.index)
returns = returns.loc[common_idx]
bench = bench.loc[common_idx]

In [5]:
#Help Functions / Risk Metrics
TRADING_DAYS = 252
RF = 0.03  # annual risk-free rate

def portfolio_returns(returns_df: pd.DataFrame, weights: dict) -> pd.Series:
    w = pd.Series(weights).reindex(returns_df.columns).fillna(0.0)
    w = w / w.sum()
    return returns_df @ w

def max_drawdown(r: pd.Series) -> float:
    wealth = (1 + r).cumprod()
    peak = wealth.cummax()
    dd = (wealth / peak) - 1
    return float(dd.min())

def var_cvar(r: pd.Series, alpha=0.05):
    # Historical VaR/CVaR on daily returns
    q = r.quantile(alpha)
    cvar = r[r <= q].mean()
    return float(q), float(cvar)

def annualized_return(r: pd.Series) -> float:
    # Geometric annualized return
    return float((1 + r).prod() ** (TRADING_DAYS / len(r)) - 1)

def annualized_vol(r: pd.Series) -> float:
    return float(r.std() * np.sqrt(TRADING_DAYS))

def sharpe_excess(r: pd.Series, rf=RF) -> float:
    ar = annualized_return(r)
    av = annualized_vol(r)
    return float((ar - rf) / av) if av != 0 else np.nan

In [6]:
# Replace these with your actual weights (as decimals)
w_max_sharpe_unconstrained = {
    "NVDA": 0.698407, "JNJ": 0.052157, "MSFT": 0.108508,
    "JPM": 0.112568, "V": 0.027972, "RTX": 0.000388
}

w_min_vol = {
    "JNJ": 0.385579, "MSFT": 0.112916, "RTX": 0.169079,
    "V": 0.225177, "JPM": 0.105910, "NVDA": 0.001339
}

# Constrained max-sharpe: plug your final constrained weights here
w_max_sharpe_constrained = {
    "NVDA": 0.399124, "MSFT": 0.245992, "JNJ": 0.213074,
    "JPM": 0.132025, "V": 0.008809, "RTX": 0.000976
}

# Equal-weight
tickers = returns.columns.tolist()
w_equal = {t: 1/len(tickers) for t in tickers}

portfolios = {
    "MaxSharpe_Uncon": w_max_sharpe_unconstrained,
    "MaxSharpe_Con40": w_max_sharpe_constrained,
    "MinVol": w_min_vol,
    "EqualWeight": w_equal
}

In [9]:
#Computing full-sample risk table
rows = []
for name, w in portfolios.items():
    r = portfolio_returns(returns, w)
    var95, cvar95 = var_cvar(r, alpha=0.05)
    var99, cvar99 = var_cvar(r, alpha=0.01)
    rows.append({
        "Portfolio": name,
        "AnnReturn": annualized_return(r),
        "AnnVol": annualized_vol(r),
        "Sharpe(excess)": sharpe_excess(r, rf=RF),
        "VaR_95(daily)": var95,
        "CVaR_95(daily)": cvar95,
        "VaR_99(daily)": var99,
        "CVaR_99(daily)": cvar99,
        "MaxDrawdown": max_drawdown(r),
        "WorstDay": float(r.min())
    })

# Add benchmark
var95, cvar95 = var_cvar(bench, alpha=0.05)
var99, cvar99 = var_cvar(bench, alpha=0.01)
rows.append({
    "Portfolio": "Benchmark_^GSPC",
    "AnnReturn": annualized_return(bench),
    "AnnVol": annualized_vol(bench),
    "Sharpe(excess)": sharpe_excess(bench, rf=RF),
    "VaR_95(daily)": var95,
    "CVaR_95(daily)": cvar95,
    "VaR_99(daily)": var99,
    "CVaR_99(daily)": cvar99,
    "MaxDrawdown": max_drawdown(bench),
    "WorstDay": float(bench.min())
})

risk_table = pd.DataFrame(rows).set_index("Portfolio").sort_values("MaxDrawdown")
risk_table

# index=False prevents pandas from adding an extra column for row numbers
risk_table.to_csv('../reports/figures/risk_table.csv', index=False)

In [10]:
#Stress test window (COVID)
stress_start, stress_end = "2020-02-15", "2020-06-30"
stress_returns = returns.loc[stress_start:stress_end]
stress_bench = bench.loc[stress_start:stress_end]

rows = []
for name, w in portfolios.items():
    r = portfolio_returns(stress_returns, w)
    var95, cvar95 = var_cvar(r, alpha=0.05)
    rows.append({
        "Portfolio": name,
        "Stress_AnnReturn": annualized_return(r),
        "Stress_AnnVol": annualized_vol(r),
        "Stress_VaR95": var95,
        "Stress_CVaR95": cvar95,
        "Stress_MaxDD": max_drawdown(r)
    })

var95, cvar95 = var_cvar(stress_bench, alpha=0.05)
rows.append({
    "Portfolio": "Benchmark_^GSPC",
    "Stress_AnnReturn": annualized_return(stress_bench),
    "Stress_AnnVol": annualized_vol(stress_bench),
    "Stress_VaR95": var95,
    "Stress_CVaR95": cvar95,
    "Stress_MaxDD": max_drawdown(stress_bench)
})

stress_table = pd.DataFrame(rows).set_index("Portfolio").sort_values("Stress_MaxDD")
stress_table

# index=False prevents pandas from adding an extra column for row numbers
stress_table.to_csv('../reports/figures/stress_table.csv', index=False)